# Importing Libraries 

In [309]:
import pandas as pd
import ast
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer 
from sklearn.metrics.pairwise import cosine_similarity

# Reading dataset

In [310]:
df = pd.read_csv('updated_aggregated_advertisers_dataset_v2.csv')

# Cleaning data 

In [311]:
def parse_industry(val):
    try:
        # Convert string like "['Home & Garden']" into a real list
        parsed = ast.literal_eval(val)
        # Join list items into a single string
        return ', '.join(parsed).lower()
    except:
        # If something goes wrong (bad format), return as is
        return val

In [312]:
def clean_goals(val):
    try:
        # Step 1: Convert outer string into actual list
        outer_list = ast.literal_eval(val)
        if isinstance(outer_list, list) and len(outer_list) > 0:
            inner_str = outer_list[0]
            # Step 2: Extract goal keywords (e.g., AWARENESS, LEADS)
            goals = re.findall(r"[A-Z_]+", inner_str)
            # Step 3: Join and lowercase
            return ", ".join(goals).lower()
        return ""
    except Exception:
        return val  # fallback if broken format

In [313]:
df['unique_industries'] = df['unique_industries'].apply(parse_industry)

In [314]:

df['unique_goals'] = df['unique_goals'].apply(clean_goals)

### just lower casing the entire data frame string values

In [315]:
df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)

/var/folders/bl/c0pvtpr55y3477p6dl_6s5kh0000gn/T/ipykernel_2248/2340421553.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


In [316]:
df['postal_codes'] = df['postal_codes'].apply(ast.literal_eval)

In [317]:
df.shape

(1390, 35)

# Creating pickle file

In [318]:
df.to_pickle('final_preprocessed_df.pkl')

# Load pickle file

In [319]:
final_preprocessed_df = pd.read_pickle('final_preprocessed_df.pkl')

# Vectorization of columns_to_use

In [320]:
columns_to_use = ['unique_goals', 'unique_industries', 'advertiser_name', 'avg_order_duration_days']

## Preprocessing column data for vectorization

In [321]:
def preprocess_text_preserve_multiword(series):
    return series.astype(str) \
        .str.replace(r"[\[\]']", "", regex=True) \
        .str.replace(r"[&]", "", regex=True) \
        .str.split(",") \
        .apply(lambda items: ' '.join(
            re.sub(r'[^\w\s]', '', item).strip().replace(" ", "_")
            for item in items if item.strip()
        ))

## Orignal df vectorization for recommendation using BOW

In [322]:
def create_semantic_bow_vectors(df):
    goal_cols = ['unique_goals']
    industry_cols = ['unique_industries']
    advertiser_name_cols = ['advertiser_name']
    
    vectors = {}
    vectorizers = {}
    
    # 1. Goals BOW Vector
    goal_text = df[goal_cols].apply(preprocess_text_preserve_multiword)
    goal_text_combined = goal_text.apply(lambda row: ' '.join(row), axis=1)
    goal_vectorizer = CountVectorizer(lowercase=True)
    goal_bow = goal_vectorizer.fit_transform(goal_text_combined)
    goal_features = [f"{name}" for name in goal_vectorizer.get_feature_names_out()]
    vectors['goals'] = pd.DataFrame(goal_bow.toarray(), columns=goal_features, index=df.index)
    vectorizers['goals'] = goal_vectorizer
    
    # 2. Industry BOW Vector
    industry_text = df[industry_cols].apply(preprocess_text_preserve_multiword)
    industry_text_combined = industry_text.apply(lambda row: ' '.join(row), axis=1)
    industry_vectorizer = CountVectorizer(lowercase=True)
    industry_bow = industry_vectorizer.fit_transform(industry_text_combined)
    industry_features = [f"{name}" for name in industry_vectorizer.get_feature_names_out()]
    vectors['industry'] = pd.DataFrame(industry_bow.toarray(), columns=industry_features, index=df.index)
    vectorizers['industry'] = industry_vectorizer
    
    # 3. Advertiser BOW Vector
    advertiser_text = df[advertiser_name_cols].apply(preprocess_text_preserve_multiword)
    advertiser_text_combined = advertiser_text.apply(lambda row: ' '.join(row), axis=1)
    advertiser_vectorizer = CountVectorizer(lowercase=True)
    advertiser_bow = advertiser_vectorizer.fit_transform(advertiser_text_combined)
    advertiser_features = [f"{name}" for name in advertiser_vectorizer.get_feature_names_out()]
    vectors['advertiser'] = pd.DataFrame(advertiser_bow.toarray(), columns=advertiser_features, index=df.index)
    vectorizers['advertiser'] = advertiser_vectorizer

    return vectors, vectorizers

# Create semantic vectors
semantic_vectors, semantic_vectorizers = create_semantic_bow_vectors(final_preprocessed_df)

# print("Geographic BOW shape:", semantic_vectors['geographic'].shape)
print("Goals BOW shape:", semantic_vectors['goals'].shape) 
print("Industry BOW shape:", semantic_vectors['industry'].shape)
print("Advertiser BOW shape:", semantic_vectors['advertiser'].shape)

# Option 1: Use vectors separately for different analyses
# geo_similarity = semantic_vectors['geographic']
goal_similarity = semantic_vectors['goals']
industry_similarity = semantic_vectors['industry']
advertiser_similarity = semantic_vectors['advertiser']

# Option 2: Combine all vectors (if needed for single model)
combined_semantic_bow = pd.concat([
    # semantic_vectors['geographic'],
    semantic_vectors['goals'], 
    semantic_vectors['industry'],
    semantic_vectors['advertiser']
], axis=1)

combined_semantic_bow_with_advertiser_name = pd.concat([
    df[['advertiser_name']],
    combined_semantic_bow
], axis=1)

Goals BOW shape: (1390, 5)
Industry BOW shape: (1390, 30)
Advertiser BOW shape: (1390, 1333)


In [323]:
combined_semantic_bow_with_advertiser_name

,advertiser_name,awareness,engagement,foot_traffic,leads,retention,agriculture,apparel__accessories,arts__entertainment,autos__vehicles,...,zamzow_mfg,zatarains,zen_windows_boston,zen_windows_cleveland,zen_windows_philadelphia,zen_windows_st_louis,zen_windows_twin_cities,zen_windows_west_michigan,zeta_charter_schools,zoofari_parks
0,n fox jewelers,1,1,1,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1 800 plumber cypress,1,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1 800 plumber lansdale,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1 800 plumber monterey,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,180 water,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1385,dataparc,1,1,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1386,ilearn schools,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1387,ilearn schools - bronx,0,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1388,lake chelan health,1,1,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Applying StandardScaler to duration days

In [324]:
preprocessor = ColumnTransformer(transformers=[
    ('duration', StandardScaler(), ['avg_order_duration_days']),
], remainder='drop')

X_transformed = preprocessor.fit_transform(final_preprocessed_df)
duration_features = ['avg_order_duration_days']

df_transformed = pd.DataFrame(X_transformed, columns=duration_features)

In [325]:
df_transformed.shape

(1390, 1)

## Concatinating both vector matrices (BOW and StandardScaler)

In [326]:
combined_semantic_bow = combined_semantic_bow.reset_index(drop=True)
df_transformed = df_transformed.reset_index(drop=True)

final_features_df = pd.concat([df_transformed, combined_semantic_bow], axis=1)

final_features_df_with_advertiser_name = pd.concat([
    df[['advertiser_name']],
    final_features_df
], axis=1)

In [327]:
final_features_df.shape

(1390, 1369)

## Save Vectorizers and Preprocessor to Avoid Reprocessing

In [328]:
# After your training code that works, add this to save all transformers:
saved_transformers = {
    'semantic_vectorizers': semantic_vectorizers,
    'preprocessor': preprocessor,
    'all_feature_names': duration_features
}
print("All transformers saved successfully!")

All transformers saved successfully!


## User side vectorization and recommending location by using cosine similarity. 

In [329]:
class LocationRecommendationSystem:
    def __init__(self, saved_transformers, feature_matrix, original_df):
        self.saved_transformers = saved_transformers
        self.feature_matrix = feature_matrix
        self.original_df = original_df
    
    def build_user_input_record(self, advertiser_name, industry, duration, goals):
        advertiser_name_cleaned = advertiser_name if advertiser_name else 'unknown_advertiser'
        industry_cleaned = industry if industry else 'unknown_industry'
        goals_cleaned = goals if goals else 'unknown_goals'
        duration_cleaned = duration if duration is not None else 0

        return pd.DataFrame([{
            'avg_order_duration_days': duration_cleaned,
            'unique_goals': goals_cleaned,
            'unique_industries': industry_cleaned,
            'advertiser_name': advertiser_name_cleaned
        }])
    
    def create_semantic_bow_vectors_inference(self, user_df):
        """Transform user input using pre-fitted vectorizers."""
        semantic_vectorizers = self.saved_transformers['semantic_vectorizers']
        
        goal_cols = ['unique_goals']
        industry_cols = ['unique_industries']
        advertiser_name_cols = ['advertiser_name']
        
        vectors = {}
        
        # Goals BOW Vector
        goal_text = user_df[goal_cols].apply(preprocess_text_preserve_multiword)
        goal_text_combined = goal_text.apply(lambda row: ' '.join(row), axis=1)
        goal_bow = semantic_vectorizers['goals'].transform(goal_text_combined)
        goal_features = [f"{name}" for name in semantic_vectorizers['goals'].get_feature_names_out()]
        vectors['goals'] = pd.DataFrame(goal_bow.toarray(), columns=goal_features, index=user_df.index)
        
        # Industry BOW Vector
        industry_text = user_df[industry_cols].apply(preprocess_text_preserve_multiword)
        industry_text_combined = industry_text.apply(lambda row: ' '.join(row), axis=1)
        industry_bow = semantic_vectorizers['industry'].transform(industry_text_combined)
        industry_features = [f"{name}" for name in semantic_vectorizers['industry'].get_feature_names_out()]
        vectors['industry'] = pd.DataFrame(industry_bow.toarray(), columns=industry_features, index=user_df.index)
        
        # Advertiser BOW Vector
        advertiser_text = user_df[advertiser_name_cols].apply(preprocess_text_preserve_multiword)
        advertiser_text_combined = advertiser_text.apply(lambda row: ' '.join(row), axis=1)
        advertiser_bow = semantic_vectorizers['advertiser'].transform(advertiser_text_combined)
        advertiser_features = [f"{name}" for name in semantic_vectorizers['advertiser'].get_feature_names_out()]
        vectors['advertiser'] = pd.DataFrame(advertiser_bow.toarray(), columns=advertiser_features, index=user_df.index)
        
        return vectors
    
    def vectorize_user_input(self, user_df):
        # Create semantic vectors
        semantic_vectors = self.create_semantic_bow_vectors_inference(user_df)
        
        # Transform other features
        X_transformed = self.saved_transformers['preprocessor'].transform(user_df)
        df_transformed = pd.DataFrame(X_transformed, columns=self.saved_transformers['all_feature_names'])
        
        # Combine semantic vectors in the same order as training
        combined_semantic_bow = pd.concat([
            semantic_vectors['goals'],
            semantic_vectors['industry'],
            semantic_vectors['advertiser']
        ], axis=1)
        
        # Reset indices and combine all features
        combined_semantic_bow = combined_semantic_bow.reset_index(drop=True)
        df_transformed = df_transformed.reset_index(drop=True)
        final_user_features = pd.concat([df_transformed, combined_semantic_bow], axis=1)
        
        return final_user_features.values
    
    def find_exact_matches(self, user_input_df, tolerance=1.0):
        user_goals = user_input_df['unique_goals'].iloc[0]
        user_industry = user_input_df['unique_industries'].iloc[0]
        user_advertiser = user_input_df['advertiser_name'].iloc[0]
        user_duration = user_input_df['avg_order_duration_days'].iloc[0]

        df = self.original_df

        # Build boolean masks for each condition
        goals_mask = df['unique_goals'] == user_goals
        industry_mask = df['unique_industries'] == user_industry
        advertiser_mask = df['advertiser_name'] == user_advertiser
        duration_mask = (df['avg_order_duration_days'] - user_duration).abs() < tolerance

        # Combine all conditions
        all_conditions = goals_mask & industry_mask & advertiser_mask & duration_mask

        # Get indices where all conditions are True
        exact_matches = df[all_conditions].index.tolist()

        return exact_matches
    
    def get_similarity_recommendations(self, user_vector, top_k=5):
        # print(user_vector.shape)
        # print(self.feature_matrix.shape)

        similarities = cosine_similarity(user_vector, self.feature_matrix)[0]
        top_indices = similarities.argsort()[::-1][:top_k]
        
        recommendations = self.original_df.iloc[top_indices][['postal_codes', 'cities', 'states']].copy()
        similarity_scores = similarities[top_indices]
        
        return recommendations, similarity_scores
    
    def recommend_locations(self, advertiser_name, industry, duration, goals, top_k=5, prefer_exact_match=True):
        # Build user input record
        user_input_df = self.build_user_input_record(advertiser_name, industry, duration, goals)
        
        # Try exact matching first
        if prefer_exact_match:
            exact_matches = self.find_exact_matches(user_input_df)
            
            if len(exact_matches) > 0:
                recommendations = self.original_df.iloc[exact_matches]['postal_codes'].head(top_k)
                all_postal_codes = []
                for codes in recommendations:
                    if isinstance(codes, list):
                        all_postal_codes.extend(codes)
                    else:
                        all_postal_codes.append(codes)

                # Remove duplicates while preserving order
                unique_postal_codes = []
                seen = set()
                for code in all_postal_codes:
                    if code not in seen:
                        unique_postal_codes.append(code)
                        seen.add(code)

                # Select top_k unique postal codes
                top_k_postal_codes = unique_postal_codes[:top_k]

                return {
                    'recommendations': top_k_postal_codes,
                    'method_used': 'exact_match',
                    'num_exact_matches': len(exact_matches),
                    'similarity_scores': None
                }
        
        # Fall back to similarity-based recommendations
        user_vector = self.vectorize_user_input(user_input_df)
        recommendations, similarity_scores = self.get_similarity_recommendations(user_vector, top_k=top_k)

        # Collect and flatten postal codes from top rows
        all_postal_codes = []
        for codes in recommendations['postal_codes']:
            if isinstance(codes, list):
                all_postal_codes.extend(codes)
            else:
                all_postal_codes.append(codes)

        # Remove duplicates while preserving order
        unique_postal_codes = []
        seen = set()
        for code in all_postal_codes:
            if code not in seen:
                unique_postal_codes.append(code)
                seen.add(code)

        # Select top_k unique postal codes
        top_k_postal_codes = unique_postal_codes[:top_k]

        return {
            'recommendations': top_k_postal_codes,
            'method_used': 'similarity_based',
            'num_exact_matches': 0,
            'similarity_scores': similarity_scores
        }


# Initialize the system with your saved transformers and data
recommendation_system = LocationRecommendationSystem(
    saved_transformers=saved_transformers,  # Your saved transformers dict
    feature_matrix=final_features_df.values,          # Your pre-computed feature matrix
    original_df=final_preprocessed_df                 # Your original training dataframe
)

## giving input


In [331]:
# Get recommendations
results = recommendation_system.recommend_locations(
    advertiser_name="n fox jewelers",
    industry='',
    duration=42,
    goals="AWARENESS, ENGAGEMENT, FOOT_TRAFFIC, LEADS, RETENTION",
    top_k=6
)

In [332]:
results['recommendations']

['12871', '12065', '12183', '12866', '12884', '12222']

## Copies the first 100 rows of the original DataFrame.
## Adds a new column to store recommended postal codes.
## Loops through each row to extract advertiser info and goals.
## Sets top_k to the number of actual postal codes in that row.
## Generates and stores recommendations using the recommendation system.


In [333]:
df_with_recommendations = df.head(100).copy()

# Create a column to store the top recommended postal code(s)
df_with_recommendations['recommended_postal_code'] = ''

# Loop through each row
for idx, row in df_with_recommendations.iterrows():
    advertiser_name = row['advertiser_name'] if 'advertiser_name' in row else ''
    industry = row['unique_industries'] if 'unique_industries' in row else ''
    duration = row['avg_order_duration_days'] if 'avg_order_duration_days' in row else 0
    goals = row['unique_goals'] if 'unique_goals' in row else ''

    actual_postal_codes = row.get('postal_codes', [])
    top_k = len(actual_postal_codes) if isinstance(actual_postal_codes, list) else 5  # fallback to 5

    # Get recommendations
    result = recommendation_system.recommend_locations(
        advertiser_name=advertiser_name,
        industry=industry,
        duration=duration,
        goals=goals,
        top_k=top_k
    )

    recommended_codes = result['recommendations']  # ✅ fixed variable name
    
    if len(recommended_codes) > 0:
        df_with_recommendations.at[idx, 'recommended_postal_code'] = recommended_codes
    else:
        df_with_recommendations.at[idx, 'recommended_postal_code'] = ''

In [334]:

df_with_recommendations

,Unnamed: 0,advertiser_name,order_goals,line_item_ids,unique_industries,unique_order_count,avg_order_duration_days,unique_goals,states,locations,...,city_count,is_focused_on_1_city,postal_code_count,is_focused_on_1_postal_code,advertiser_postal_code,advertiser_city,advertiser_state_code,runs_in_business_state,runs_in_business_city,recommended_postal_code
0,0,n fox jewelers,['awareness' 'engagement' 'foot_traffic' 'lead...,"[3886504, 3886504, 3886504, 3886504, 3886504, ...",apparel & accessories,2,42.514286,"awareness, engagement, foot_traffic, leads, re...",['ny'],"['fort edward, new york, us', 'ballston lake, ...",...,38,0,44,0,12866.0,saratoga springs,ny,1,1,"[12871, 12065, 12183, 12866, 12884, 12222, 120..."
1,1,1 800 plumber cypress,['awareness' 'engagement' 'leads' 'retention'],"[7532319, 7532319, 7532319, 7532319, 7532319, ...",home & garden,1,337.000000,"awareness, engagement, leads, retention",['tx'],"['conroe, texas, us', 'houston, texas, us', 'c...",...,5,0,14,0,77429.0,cypress,tx,1,1,"[77388, 77433, 77386, 77373, 77429, 77380, 773..."
2,2,1 800 plumber lansdale,['awareness'],"[3426684, 3426684, 3426684, 3426684, 3426684, ...",home & garden,1,121.000000,awareness,['pa'],"['willow grove, pennsylvania, us', 'blue bell,...",...,43,0,44,0,19446.0,lansdale,pa,1,1,"[19002, 19004, 19456, 19446, 19436, 19405, 190..."
3,3,1 800 plumber monterey,['engagement' 'leads' 'retention'],"[4493331, 4493331, 4493331, 4493331, 4493331, ...",home & garden,1,80.000000,"engagement, leads, retention",['ca'],"['moss landing, california, us', 'santa cruz, ...",...,21,0,28,0,93940.0,monterey,ca,1,0,"[95066, 95004, 95073, 95007, 95019, 93905, 950..."
4,4,180 water,['engagement' 'leads' 'retention'],"[4963424, 4963424, 4963424, 4963424, 4963424, ...",home & garden,1,227.000000,"engagement, leads, retention","['id', 'mt', 'wy']","['wolf, wyoming, us', 'lovell, wyoming, us', '...",...,208,0,244,0,10038.0,new york,ny,0,0,"[59018, 59720, 59739, 59450, 82423, 59019, 590..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,bazonzoes,['awareness' 'engagement' 'foot_traffic' 'lead...,"[6756292, 6756292, 6756292, 6756292, 6756292, ...",cannabis/cbd,1,149.000000,"awareness, engagement, foot_traffic, leads, re...",['mi'],"['dearborn, michigan, us', 'highland, michigan...",...,87,0,171,0,48917.0,lansing,mi,1,1,"[48894, 48034, 48930, 48071, 48139, 48918, 488..."
96,96,beamer group,['awareness' 'engagement' 'foot_traffic' 'lead...,"[4711389, 4711389, 4711389, 4711389, 4711389, ...",health,1,9.000000,"awareness, engagement, foot_traffic, leads, re...",['ga'],"['ideal, georgia, us', 'byron, georgia, us', '...",...,37,0,61,0,31025.0,macon,ga,1,1,"[31295, 31091, 31016, 31806, 31221, 31052, 317..."
97,97,bell tower theatre,['awareness' 'engagement' 'leads' 'retention'],"[6535716, 6535716, 6535716, 6535716, 6535716, ...",arts & entertainment,1,177.000000,"awareness, engagement, leads, retention","['wi', 'il', 'ia']","['sinsinawa, wisconsin, us', 'dyersville, iowa...",...,46,0,50,0,52001.0,dubuque,ia,1,1,"[53818, 52041, 52040, 61075, 52031, 52056, 538..."
98,98,benjamin plumbing,['engagement' 'leads' 'retention'],"[11364739, 11364739, 11364739, 11364739, 11364...",home & garden,1,7.000000,"engagement, leads, retention",['wi'],"['orfordville, wisconsin, us', 'madison, wisco...",...,121,0,161,0,53719.0,madison,wi,1,1,"[53501, 53595, 53933, 53533, 53582, 53532, 535..."


## matching the postal codes with recommended postal codes

In [304]:
def all_recommended_in_postal_codes(row):
    recommended = row['recommended_postal_code']
    actual = row['postal_codes']
    
    if isinstance(recommended, list) and isinstance(actual, list):
        return all(code in actual for code in recommended)
    return False

df_with_recommendations['is_match'] = df_with_recommendations.apply(all_recommended_in_postal_codes, axis=1)


## giving accuracy in percentage

In [305]:
accuracy = df_with_recommendations['is_match'].mean()
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 100.00%
